# V22 temporal U-Net detection-availability shadow

Runs the frozen 12-event detector-only battery. It does not infer edges or mutate an Atabey graph. Kaggle GPU is preferred because the competition data and public support pack can be mounted directly.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IS_KAGGLE = Path('/kaggle/input').exists()
if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/Atabey')
    TRAIN_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
    SUPPORT_ROOT = Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1')
    if not SUPPORT_ROOT.exists():
        SUPPORT_ROOT = Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')
else:
    PROJECT_ROOT = Path(os.environ.get('ATABEY_ROOT', '/content/Atabey'))
    TRAIN_DIR = Path(os.environ.get('ATABEY_TRAIN_DIR', '/content/Atabey/train'))
    SUPPORT_ROOT = Path(os.environ.get('BIOHUB_SUPPORT_ROOT', '/content/biohub-tracking-support-pack-50ep-v1'))

print('runtime:', 'Kaggle' if IS_KAGGLE else 'Colab/local')
print('project:', PROJECT_ROOT)
print('train:', TRAIN_DIR, TRAIN_DIR.exists())
print('support:', SUPPORT_ROOT, SUPPORT_ROOT.exists())
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))

Attach these Kaggle inputs before running: the competition data and `pilkwang/biohub-tracking-support-pack-50ep-v1`. The Atabey branch must already exist at `PROJECT_ROOT`; cloning requires an internet-enabled research session.

In [ ]:
if not PROJECT_ROOT.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'v22-upstream-division-availability', '--single-branch',
        'https://github.com/drosadocastro-bit/Atabey.git', str(PROJECT_ROOT)
    ], check=True)

requirements = SUPPORT_ROOT / 'requirements-unet-ilp-kaggle-predownload.txt'
wheels = SUPPORT_ROOT / 'wheels'
if requirements.exists() and wheels.exists():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
        '--find-links', str(wheels), '-r', str(requirements)
    ], check=True)
else:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'zarr>=3.0.10,<4', 'polars>=1.36', 'tracksdata', 'geff>=1.1.3.1.1'
    ], check=True)

assert (PROJECT_ROOT / 'scripts/run_v22_unet_detection_shadow.py').exists()
assert TRAIN_DIR.exists()
assert (SUPPORT_ROOT / 'repo/scripts/predict_unet_transformer.py').exists()
assert (SUPPORT_ROOT / 'weights/unet_transformer/split_0/edge_predictor_best.pth').exists()
print('Environment ready')

In [ ]:
output_csv = PROJECT_ROOT / 'v22_unet_detection_shadow.csv'
output_summary = PROJECT_ROOT / 'v22_unet_detection_shadow_summary.json'
cmd = [
    sys.executable, '-u', str(PROJECT_ROOT / 'scripts/run_v22_unet_detection_shadow.py'),
    '--train-dir', str(TRAIN_DIR),
    '--support-repo', str(SUPPORT_ROOT / 'repo'),
    '--weights', str(SUPPORT_ROOT / 'weights/unet_transformer/split_0/edge_predictor_best.pth'),
    '--fixture', str(PROJECT_ROOT / 'tests/fixtures/v22_unet_detection_shadow.json'),
    '--output-csv', str(output_csv),
    '--output-summary', str(output_summary),
]
env = {**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')}
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)

In [ ]:
import json, pandas as pd
display(pd.read_csv(output_csv))
print(json.dumps(json.loads(output_summary.read_text()), indent=2))
print('Download both output files before ending the runtime.')